In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor

# -------------------------------------------------
# Paths (package style)
# -------------------------------------------------
ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results" / "intermediate"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TERRAIN_PATH = DATA_DIR / "weightedTerrainData.csv"

OUT_LONG = RESULTS_DIR / "Table2_P_XGBoost_detail.csv"
OUT_SUMM = RESULTS_DIR / "Table2_P_XGBoost_summary.csv"

# -------------------------------------------------
# IDs
# -------------------------------------------------
TURBINE_IDS = list(range(1, 67))
TESTSET_2018 = list(range(1, 47)) + [48, 49, 50, 52] + list(range(54, 61)) + list(range(62, 67))

# -------------------------------------------------
# Terrain
# -------------------------------------------------
terrain_df = pd.read_csv(TERRAIN_PATH)
terrain_cols = terrain_df.columns[1:4]
terrain_mat = terrain_df.loc[:65, terrain_cols].values

# -------------------------------------------------
# Helpers
# -------------------------------------------------
def rmse(yhat, y):
    yhat = np.asarray(yhat, float)
    y = np.asarray(y, float)
    m = np.isfinite(yhat) & np.isfinite(y)
    return float(np.sqrt(np.mean((yhat[m] - y[m])**2))) if np.any(m) else np.nan


def load_turbine_csv(tid, year):
    f = DATA_DIR / f"Turbine{tid}_{year}.csv"
    if not f.exists():
        return None

    df = pd.read_csv(f)

    needed = [
        "wind_speed",
        "temperature",
        "turbulence_intensity",
        "std_wind_direction",
        "wind_direction",
        "power",
    ]

    df = df[needed].copy()

    for c in needed:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna()

    if df.empty:
        return None

    # -------- sin/cos transformation --------
    rad = np.deg2rad(df["wind_direction"].values)
    df["wind_direction_sin"] = np.sin(rad)
    df["wind_direction_cos"] = np.cos(rad)

    df = df.drop(columns=["wind_direction"])

    return df


FEATURES = [
    "wind_speed",
    "temperature",
    "turbulence_intensity",
    "std_wind_direction",
    "wind_direction_sin",
    "wind_direction_cos",
]


# -------------------------------------------------
# Cache data
# -------------------------------------------------
data_cache = {}
for tid in TURBINE_IDS:
    for yr in (2017, 2018):
        data_cache[(tid, yr)] = load_turbine_csv(tid, yr)

# -------------------------------------------------
# Build pooled 2017 dataset
# -------------------------------------------------
X2017_list, y2017_list, tid2017_list = [], [], []

for tid in TURBINE_IDS:
    df = data_cache[(tid, 2017)]
    if df is None:
        continue

    X = df[FEATURES].values
    y = df["power"].values
    t = np.full(len(y), tid, dtype=int)

    X2017_list.append(X)
    y2017_list.append(y)
    tid2017_list.append(t)

X2017 = np.vstack(X2017_list)
y2017 = np.concatenate(y2017_list)
tid2017 = np.concatenate(tid2017_list)

S2017 = terrain_mat[tid2017 - 1]

# -------------------------------------------------
# Model
# -------------------------------------------------
def fit_lgbm(X_train, y_train, X_test, seed):
    model = LGBMRegressor(
        objective="regression",
        n_estimators=200,
        learning_rate=0.1,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    return model.predict(X_test)


# -------------------------------------------------
# Main loop
# -------------------------------------------------
results = []

for year in (2017, 2018):
    test_ids = TURBINE_IDS if year == 2017 else TESTSET_2018

    for i in test_ids:
        print(f"Evaluating Turbine {i}, Year {year}")

        df_test = data_cache[(i, year)]
        if df_test is None:
            continue

        X_test = df_test[FEATURES].values
        y_test = df_test["power"].values

        S_test = np.tile(terrain_mat[i - 1], (len(X_test), 1))

        mask = tid2017 != i
        X_train = X2017[mask]
        y_train = y2017[mask]
        S_train = S2017[mask]

        # -----------------
        # LGBM(x)
        # -----------------
        t0 = time.time()
        pred_x = fit_lgbm(X_train, y_train, X_test, seed=i)
        runtime = time.time() - t0

        results.append({
            "method": "P_XGBoost(x)",
            "target": i,
            "year": year,
            "rmse": rmse(pred_x, y_test),
            "runtime_sec": runtime,
        })

        # -----------------
        # LGBM(x+s)
        # -----------------
        Xs_train = np.hstack([X_train, S_train])
        Xs_test = np.hstack([X_test, S_test])

        t0 = time.time()
        pred_xs = fit_lgbm(Xs_train, y_train, Xs_test, seed=i)
        runtime = time.time() - t0

        results.append({
            "method": "P_XGBoost(x+s)",
            "target": i,
            "year": year,
            "rmse": rmse(pred_xs, y_test),
            "runtime_sec": runtime,
        })

# -------------------------------------------------
# Save
# -------------------------------------------------
detail_df = pd.DataFrame(results)
detail_df.to_csv(OUT_LONG, index=False)

summary_df = (
    detail_df
    .groupby(["method", "year"], as_index=False)
    .agg(
        avg_rmse=("rmse", "mean"),
        total_runtime_sec=("runtime_sec", "sum"),
    )
)

summary_df.to_csv(OUT_SUMM, index=False)

print("Saved:")
print(OUT_LONG)
print(OUT_SUMM)